In [1]:
# ════════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# ════════════════════════════════════════════════════════════════════
import os, gc, time, random, warnings
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.special import logit
from scipy.stats import rankdata
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder, KBinsDiscretizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier

def seed_everything(s=42):
    random.seed(s); np.random.seed(s)

seed_everything(42)

# ════════════════════════════════════════════════════════════════════
# CELL 2 — CONFIG & PATHS
# ════════════════════════════════════════════════════════════════════
TARGET  = 'Churn'
ID_COL  = 'id'
N_FOLDS = 10
SEED    = 42

TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e3/test.csv'
ORIG_PATH  = '/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv'

# ── LOAD v7 OOF PREDICTIONS (saved by v7 notebook) ──────────────────
# Ensure you have added the output of your v7 notebook as "Data" to this notebook
V7_OOF_DIR = '/kaggle/input/notebooks/akalyyanahmed/s6e3-v7/'

V7_MODEL_NAMES = ['lgbm_a', 'lgbm_b', 'xgb', 'catboost',
                   'ridge_ohe', 'realmlp', 'tabm']

print("Loading v7 OOF predictions...")
oof_v7   = {}
preds_v7 = {}
for mn in V7_MODEL_NAMES:
    oof_path  = os.path.join(V7_OOF_DIR, f'oof_v7_{mn}.npy')
    test_path = os.path.join(V7_OOF_DIR, f'test_v7_{mn}.npy')
    
    if os.path.exists(oof_path):
        oof_v7[mn]   = np.load(oof_path)
        preds_v7[mn] = np.load(test_path)
        print(f"  ✓ {mn}: Loaded successfully") 
    else:
        print(f"  ✗ {mn}: NOT FOUND at {oof_path}")

CATS = ['gender','SeniorCitizen','Partner','Dependents','PhoneService','MultipleLines',
        'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
        'TechSupport','StreamingTV','StreamingMovies','Contract',
        'PaperlessBilling','PaymentMethod']
NUMS = ['tenure', 'MonthlyCharges', 'TotalCharges']
TOP_CATS_NGRAM = ['Contract','InternetService','PaymentMethod',
                  'OnlineSecurity','TechSupport','PaperlessBilling']

# ════════════════════════════════════════════════════════════════════
# CELL 3 — RELOAD DATA AND FEATURES (same as v7)
# ════════════════════════════════════════════════════════════════════
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
orig  = pd.read_csv(ORIG_PATH).drop(columns=['customerID'], errors='ignore')

train[TARGET] = (train[TARGET] == 'Yes').astype(int)
orig[TARGET]  = (orig[TARGET] == 'Yes').astype(int)
orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
orig['TotalCharges'].fillna(orig['TotalCharges'].median(), inplace=True)

train_ids = train[ID_COL].copy()
test_ids  = test[ID_COL].copy()
y = train[TARGET].values
n_train, n_test = len(train), len(test)

# Now load v7 OOF and print actual AUCs
print("\nv7 model OOF AUCs:")
for mn in list(oof_v7.keys()):
    auc = roc_auc_score(y, oof_v7[mn])
    print(f"  {mn:<15}: {auc:.5f}")

# ════════════════════════════════════════════════════════════════════
# CELL 4 — FEATURE ENGINEERING (same minimal set as v7)
# ════════════════════════════════════════════════════════════════════
NEW_NUMS, NUM_AS_CAT, NGRAM_COLS = [], [], []

global_mean = orig[TARGET].mean()
for col in CATS + NUMS:
    stats = orig.groupby(col)[TARGET].mean().rename(f'{col}_org_mean')
    train = train.merge(stats.reset_index(), on=col, how='left')
    test  = test.merge(stats.reset_index(),  on=col, how='left')
    for df in [train, test]:
        df[f'{col}_org_mean'] = df[f'{col}_org_mean'].fillna(global_mean).astype('float32')
    NEW_NUMS.append(f'{col}_org_mean')

for col in NUMS:
    freq = pd.concat([train[col], orig[col], test[col]]).value_counts(normalize=True)
    for df in [train, test]:
        df[f'FREQ_{col}'] = df[col].map(freq).fillna(0).astype('float32')
    NEW_NUMS.append(f'FREQ_{col}')

for df in [train, test]:
    t = df['tenure'].clip(lower=1)
    df['charges_deviation']      = (df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']).astype('float32')
    df['monthly_to_total_ratio'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1)).astype('float32')
    df['avg_monthly_charges']    = (df['TotalCharges'] / t).astype('float32')
    SVC = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
           'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    df['service_count']  = (df[SVC] == 'Yes').sum(axis=1).astype('float32')
    df['tenure_x_mc']    = (df['tenure'] * df['MonthlyCharges']).astype('float32')
NEW_NUMS += ['charges_deviation','monthly_to_total_ratio','avg_monthly_charges',
             'service_count','tenure_x_mc']

def pctrank(v, ref): return (np.searchsorted(np.sort(ref), v) / len(ref)).astype('float32')
c_tc  = orig.loc[orig[TARGET]==1, 'TotalCharges'].values
nc_tc = orig.loc[orig[TARGET]==0, 'TotalCharges'].values
for df in [train, test]:
    tc = df['TotalCharges'].values
    df['pctrank_churner_TC']    = pctrank(tc, c_tc)
    df['pctrank_nonchurner_TC'] = pctrank(tc, nc_tc)
NEW_NUMS += ['pctrank_churner_TC','pctrank_nonchurner_TC']

for col in NUMS:
    _new = f'CAT_{col}'
    NUM_AS_CAT.append(_new)
    for df in [train, test]:
        df[_new] = df[col].astype(str).astype('category')

for col, n_bins in [('TotalCharges',4000),('TotalCharges',500),
                    ('MonthlyCharges',200),('MonthlyCharges',100)]:
    bname = f'{col}_{n_bins}_bin'
    kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
    train[bname] = kb.fit_transform(train[[col]]).ravel().astype('int32')
    test[bname]  = kb.transform(test[[col]]).ravel().astype('int32')
    for df in [train, test]:
        df[bname] = df[bname].astype('category')
    NUM_AS_CAT.append(bname)

for c1, c2 in combinations(TOP_CATS_NGRAM, 2):
    cn = f'BG_{c1}_{c2}'
    for df in [train, test]:
        df[cn] = (df[c1].astype(str) + '_' + df[c2].astype(str)).astype('category')
    NGRAM_COLS.append(cn)
TOP4 = TOP_CATS_NGRAM[:4]
for c1, c2, c3 in combinations(TOP4, 3):
    cn = f'TG_{c1}_{c2}_{c3}'
    for df in [train, test]:
        df[cn] = (df[c1].astype(str) + '_' + df[c2].astype(str) + '_' + df[c3].astype(str)).astype('category')
    NGRAM_COLS.append(cn)

FEATURES  = NUMS + CATS + NEW_NUMS + NUM_AS_CAT + NGRAM_COLS
TE_COLS   = NUM_AS_CAT + CATS
TE_NGRAM  = NGRAM_COLS
TO_REMOVE = NUM_AS_CAT + CATS + NGRAM_COLS
STATS     = ['std','min','max']
print(f"Features: {len(FEATURES)}")

# ════════════════════════════════════════════════════════════════════
# CELL 5 — TE HELPER
# ════════════════════════════════════════════════════════════════════
def apply_te(X_tr, y_tr, X_val, X_te, te_cols, te_ngram, stats, inner_folds=5, seed=42):
    skf_i = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=seed)
    for col in te_cols:
        for s in stats:
            X_tr[f'TE1_{col}_{s}'] = 0.0
    for col in te_ngram:
        X_tr[f'TE_ng_{col}'] = 0.5

    for _, (in_tr, in_va) in enumerate(skf_i.split(X_tr, y_tr)):
        X_tr2 = X_tr.loc[in_tr].copy()
        for col in te_cols:
            tmp = X_tr2.groupby(col, observed=False)[TARGET].agg(stats)
            tmp.columns = [f'TE1_{col}_{s}' for s in stats]
            for c in tmp.columns:
                X_tr.loc[in_va, c] = pd.to_numeric(
                    X_tr.loc[in_va, col].astype(str).map(tmp[c]),
                    errors='coerce').fillna(0).values.astype('float32')
        for col in te_ngram:
            ng_te = X_tr2.groupby(col, observed=False)[TARGET].mean()
            X_tr.loc[in_va, f'TE_ng_{col}'] = pd.to_numeric(
                X_tr.loc[in_va, col].astype(str).map(ng_te),
                errors='coerce').fillna(0.5).values.astype('float32')

    for col in te_cols:
        tmp = X_tr.groupby(X_tr[col].astype(str), observed=False)[TARGET].agg(stats)
        tmp.columns = [f'TE1_{col}_{s}' for s in stats]
        for c in tmp.columns:
            X_val[c] = X_val[col].astype(str).map(tmp[c]).fillna(0).astype('float32')
            X_te[c]  = X_te[col].astype(str).map(tmp[c]).fillna(0).astype('float32')
            X_tr[c]  = X_tr[c].fillna(0).astype('float32')
    for col in te_ngram:
        ng_te = X_tr.groupby(X_tr[col].astype(str), observed=False)[TARGET].mean()
        ng_name = f'TE_ng_{col}'
        X_val[ng_name] = pd.to_numeric(X_val[col].astype(str).map(ng_te), errors='coerce').fillna(0.5).astype('float32')
        X_te[ng_name]  = pd.to_numeric(X_te[col].astype(str).map(ng_te), errors='coerce').fillna(0.5).astype('float32')

    te_enc = TargetEncoder(cv=inner_folds, shuffle=True, smooth='auto', target_type='binary', random_state=seed)
    TE_MEAN_COLS = [f'TE_{col}' for col in te_cols]
    X_tr[TE_MEAN_COLS]  = te_enc.fit_transform(X_tr[te_cols].astype(str), y_tr)
    X_val[TE_MEAN_COLS] = te_enc.transform(X_val[te_cols].astype(str))
    X_te[TE_MEAN_COLS]  = te_enc.transform(X_te[te_cols].astype(str))

    for df in [X_tr, X_val, X_te]:
        for c in CATS + NUM_AS_CAT:
            if c in df.columns:
                df[c] = df[c].astype(str).astype('category')
        df.drop(columns=[c for c in TO_REMOVE if c in df.columns], inplace=True, errors='ignore')
    X_tr.drop(columns=[TARGET], inplace=True, errors='ignore')
    return X_tr, X_val, X_te

# ════════════════════════════════════════════════════════════════════
# CELL 6 — NEW MODELS: HGB + MULTI-SEED XGB
# ════════════════════════════════════════════════════════════════════
print(f"\n{'='*55}")
print("TRAINING NEW MODELS")
print(f"{'='*55}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# New model storage
new_models = ['hgb', 'xgb_seed2', 'xgb_seed3', 'lgbm_seed2']
oof_new   = {mn: np.zeros(n_train) for mn in new_models}
preds_new = {mn: np.zeros(n_test)  for mn in new_models}
fold_scores_new = {mn: [] for mn in new_models}

HGB_PARAMS = {
    'max_iter': 20000, 'random_state': SEED,
    'early_stopping': True,
    'categorical_features': 'from_dtype',
    'learning_rate': 0.01, 'loss': 'log_loss', 'scoring': 'loss',
    'l2_regularization': 1.72e-9,
    'max_depth': 6, 'max_leaf_nodes': 110, 'min_samples_leaf': 45,
}

XGB_SEED2 = {
    'n_estimators': 50000, 'learning_rate': 0.0063,
    'max_depth': 5, 'subsample': 0.81, 'colsample_bytree': 0.32,
    'min_child_weight': 6, 'reg_alpha': 3.5017, 'reg_lambda': 1.2925,
    'gamma': 0.790, 'random_state': 1337,  # different seed!
    'early_stopping_rounds': 500, 'objective': 'binary:logistic',
    'eval_metric': 'auc', 'enable_categorical': True,
    'device': 'cuda', 'tree_method': 'hist', 'verbosity': 0,
}
XGB_SEED3 = {**XGB_SEED2, 'random_state': 2024}

LGBM_SEED2 = {
    'n_estimators': 20000, 'learning_rate': 0.00833,
    'max_depth': 7, 'num_leaves': 77,
    'reg_alpha': 3.05, 'reg_lambda': 0.225,
    'min_child_samples': 56, 'subsample': 0.675,
    'colsample_bytree': 0.646, 'min_split_gain': 0.076,
    'extra_trees': True,
    'random_state': 1337,  # different seed!
    'objective': 'binary', 'metric': 'auc',
    'device': 'gpu', 'verbose': -1, 'n_jobs': -1,
}

t0 = time.time()

for i, (tr_idx, val_idx) in enumerate(skf.split(train, y)):
    print(f"\nFold {i+1}/{N_FOLDS}  ({(time.time()-t0)/60:.1f}m)")

    X_tr_raw  = train.loc[tr_idx, FEATURES + [TARGET]].reset_index(drop=True).copy()
    y_tr      = y[tr_idx]
    X_val_raw = train.loc[val_idx, FEATURES].reset_index(drop=True).copy()
    y_val     = y[val_idx]
    X_te_raw  = test[FEATURES].reset_index(drop=True).copy()

    X_tr, X_val, X_te = apply_te(X_tr_raw, y_tr, X_val_raw, X_te_raw,
                                  TE_COLS, TE_NGRAM, STATS)
    COLS = X_tr.columns

    # ── HistGradientBoosting ─────────────────────────────────────────
    # Cast category cols for HGB
    X_tr_hgb  = X_tr.copy()
    X_val_hgb = X_val.copy()
    X_te_hgb  = X_te[COLS].copy()
    for c in COLS:
        if X_tr_hgb[c].dtype.name == 'category':
            X_tr_hgb[c]  = X_tr_hgb[c].astype('category')
            X_val_hgb[c] = X_val_hgb[c].astype('category')
            X_te_hgb[c]  = X_te_hgb[c].astype('category')

    m = HistGradientBoostingClassifier(**HGB_PARAMS)
    m.fit(X_tr_hgb, y_tr)
    oof_new['hgb'][val_idx]  = m.predict_proba(X_val_hgb)[:, 1]
    preds_new['hgb']        += m.predict_proba(X_te_hgb)[:, 1] / N_FOLDS
    auc_hgb = roc_auc_score(y_val, oof_new['hgb'][val_idx])
    fold_scores_new['hgb'].append(auc_hgb)
    print(f"  HGB AUC: {auc_hgb:.5f}")
    del m; gc.collect()

    # ── XGB seed 2 ───────────────────────────────────────────────────
    m = XGBClassifier(**XGB_SEED2)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_new['xgb_seed2'][val_idx]  = m.predict_proba(X_val)[:, 1]
    preds_new['xgb_seed2']        += m.predict_proba(X_te[COLS])[:, 1] / N_FOLDS
    auc_xgb2 = roc_auc_score(y_val, oof_new['xgb_seed2'][val_idx])
    fold_scores_new['xgb_seed2'].append(auc_xgb2)
    print(f"  XGB(seed=1337) AUC: {auc_xgb2:.5f}")
    del m; gc.collect()

    # ── XGB seed 3 ───────────────────────────────────────────────────
    m = XGBClassifier(**XGB_SEED3)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_new['xgb_seed3'][val_idx]  = m.predict_proba(X_val)[:, 1]
    preds_new['xgb_seed3']        += m.predict_proba(X_te[COLS])[:, 1] / N_FOLDS
    auc_xgb3 = roc_auc_score(y_val, oof_new['xgb_seed3'][val_idx])
    fold_scores_new['xgb_seed3'].append(auc_xgb3)
    print(f"  XGB(seed=2024) AUC: {auc_xgb3:.5f}")
    del m; gc.collect()

    # ── LGBM extra_trees seed 2 ──────────────────────────────────────
    m = LGBMClassifier(**LGBM_SEED2)
    m.fit(pd.DataFrame(X_tr.values, columns=COLS), y_tr,
          eval_set=[(pd.DataFrame(X_val.values, columns=COLS), y_val)],
          callbacks=[early_stopping(200, verbose=False), log_evaluation(-1)])
    oof_new['lgbm_seed2'][val_idx]  = m.predict_proba(pd.DataFrame(X_val.values, columns=COLS))[:, 1]
    preds_new['lgbm_seed2']        += m.predict_proba(pd.DataFrame(X_te.values, columns=COLS))[:, 1] / N_FOLDS
    auc_lgbm2 = roc_auc_score(y_val, oof_new['lgbm_seed2'][val_idx])
    fold_scores_new['lgbm_seed2'].append(auc_lgbm2)
    print(f"  LGBM-ET(seed=1337) AUC: {auc_lgbm2:.5f}")
    del m; gc.collect()

print(f"\n{'='*50}")
print("NEW MODEL SUMMARY")
for mn in new_models:
    cv  = roc_auc_score(y, oof_new[mn])
    avg = np.mean(fold_scores_new[mn])
    print(f"  {mn:<15}: OOF={cv:.5f} | Mean={avg:.5f}")

# ════════════════════════════════════════════════════════════════════
# CELL 7 — COMBINED META-LEARNER (v7 models + new models)
# ════════════════════════════════════════════════════════════════════
print(f"\n{'='*55}")
print("COMBINED META-LEARNER (v7 + new models)")
print(f"{'='*55}")

# Combine all OOF
all_names = list(oof_v7.keys()) + new_models
all_oof   = [oof_v7[mn]   for mn in oof_v7] + [oof_new[mn]   for mn in new_models]
all_preds = [preds_v7[mn] for mn in preds_v7] + [preds_new[mn] for mn in new_models]

print(f"\nAll model OOF AUCs ({len(all_names)} models):")
for nm, oof_arr in zip(all_names, all_oof):
    print(f"  {nm:<20}: {roc_auc_score(y, oof_arr):.5f}")

# Meta features with logit transform
eps = 1e-7
meta_tr = logit(np.column_stack([np.clip(o, eps, 1-eps) for o in all_oof]))
meta_te = logit(np.column_stack([np.clip(p, eps, 1-eps) for p in all_preds]))

oof_meta  = np.zeros(n_train)
pred_meta = np.zeros(n_test)

for i, (tr_idx, val_idx) in enumerate(skf.split(meta_tr, y)):
    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    lr.fit(meta_tr[tr_idx], y[tr_idx])
    oof_meta[val_idx] = lr.predict_proba(meta_tr[val_idx])[:, 1]
    pred_meta        += lr.predict_proba(meta_te)[:, 1] / N_FOLDS
    
    print(f"  Meta Fold {i+1}: {roc_auc_score(y[val_idx], oof_meta[val_idx]):.5f}")

meta_cv = roc_auc_score(y, oof_meta)
print(f"\n★ Combined Meta CV AUC: {meta_cv:.5f}")

# ════════════════════════════════════════════════════════════════════
# CELL 8 — SUBMISSION
# ════════════════════════════════════════════════════════════════════
final = np.clip(pred_meta, 1e-6, 1-1e-6)
sub   = pd.DataFrame({'id': test_ids, TARGET: final})
sub.to_csv('submission_v8.csv', index=False)

print(f"\n{'='*55}")
print(f"  SUBMISSION v8 READY")
print(f"  Combined Meta CV: {meta_cv:.5f}")
print(f"  Pred mean: {final.mean():.4f}")
print(sub.head(5).to_string())
print(f"\n✓ submission_v8.csv saved!")

Loading v7 OOF predictions...
  ✓ lgbm_a: Loaded successfully
  ✓ lgbm_b: Loaded successfully
  ✓ xgb: Loaded successfully
  ✓ catboost: Loaded successfully
  ✓ ridge_ohe: Loaded successfully
  ✓ realmlp: Loaded successfully
  ✓ tabm: Loaded successfully

v7 model OOF AUCs:
  lgbm_a         : 0.91908
  lgbm_b         : 0.91900
  xgb            : 0.91922
  catboost       : 0.91884
  ridge_ohe      : 0.91050
  realmlp        : 0.91890
  tabm           : 0.91848
Features: 74

TRAINING NEW MODELS

Fold 1/10  (0.0m)
  HGB AUC: 0.91802
  XGB(seed=1337) AUC: 0.91918
  XGB(seed=2024) AUC: 0.91915


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  LGBM-ET(seed=1337) AUC: 0.91889

Fold 2/10  (19.1m)
  HGB AUC: 0.91754
  XGB(seed=1337) AUC: 0.91815
  XGB(seed=2024) AUC: 0.91820
  LGBM-ET(seed=1337) AUC: 0.91791

Fold 3/10  (37.1m)
  HGB AUC: 0.91949
  XGB(seed=1337) AUC: 0.92033
  XGB(seed=2024) AUC: 0.92036
  LGBM-ET(seed=1337) AUC: 0.92011

Fold 4/10  (54.6m)
  HGB AUC: 0.91764
  XGB(seed=1337) AUC: 0.91834
  XGB(seed=2024) AUC: 0.91835
  LGBM-ET(seed=1337) AUC: 0.91796

Fold 5/10  (72.5m)
  HGB AUC: 0.91743
  XGB(seed=1337) AUC: 0.91847
  XGB(seed=2024) AUC: 0.91848
  LGBM-ET(seed=1337) AUC: 0.91815

Fold 6/10  (87.3m)
  HGB AUC: 0.91833
  XGB(seed=1337) AUC: 0.91882
  XGB(seed=2024) AUC: 0.91886
  LGBM-ET(seed=1337) AUC: 0.91854

Fold 7/10  (101.6m)
  HGB AUC: 0.91968
  XGB(seed=1337) AUC: 0.92073
  XGB(seed=2024) AUC: 0.92076
  LGBM-ET(seed=1337) AUC: 0.92049

Fold 8/10  (118.2m)
  HGB AUC: 0.91850
  XGB(seed=1337) AUC: 0.91922
  XGB(seed=2024) AUC: 0.91924
  LGBM-ET(seed=1337) AUC: 0.91902

Fold 9/10  (136.3m)
  HGB AUC: 0